In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('../data/filtered/Fil_SCF_20022.csv', index_col=0)

In [3]:
df.columns

Index(['RETQLIQ', 'INCOME', 'FIN', 'NFIN', 'DEBT', 'EQUITY', 'STOCKS',
       'HHOUSES', 'LIQ', 'YESFINRISK', 'SPENDMOR', 'LATE', 'EMERGSAV',
       'FINLIT', 'AGE', 'EDUC', 'MARRIED', 'KIDS', 'OCCAT1', 'BUS'],
      dtype='str')

In [4]:
df['LOG_RETQLIQ'] = np.log1p(df['RETQLIQ'])
df = df.drop(['RETQLIQ'], axis=1)
df.shape

(4595, 20)

In [5]:
corr = df.corr(numeric_only=True)
corr['LOG_RETQLIQ'].sort_values(ascending=False)

LOG_RETQLIQ    1.000000
EDUC           0.499085
HHOUSES        0.430080
EMERGSAV       0.407986
FINLIT         0.370634
AGE            0.162089
INCOME         0.121472
NFIN           0.120881
FIN            0.116787
DEBT           0.103407
BUS            0.102698
EQUITY         0.097160
LIQ            0.081561
STOCKS         0.056341
YESFINRISK    -0.000706
SPENDMOR      -0.018398
KIDS          -0.049923
OCCAT1        -0.107909
LATE          -0.181822
MARRIED       -0.343269
Name: LOG_RETQLIQ, dtype: float64

In [6]:
X = df.drop(columns=['LOG_RETQLIQ'])
y = df['LOG_RETQLIQ']

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

EQUITY        0.718186
FIN           0.084791
LIQ           0.033959
STOCKS        0.031405
AGE           0.024193
INCOME        0.024180
NFIN          0.021340
DEBT          0.012622
EDUC          0.010807
BUS           0.009426
SPENDMOR      0.006742
OCCAT1        0.005198
KIDS          0.004390
FINLIT        0.004305
MARRIED       0.003287
EMERGSAV      0.001816
LATE          0.001289
HHOUSES       0.001052
YESFINRISK    0.001010
dtype: float64

In [7]:
result = permutation_importance(model, X, y, n_repeats=10, random_state=42)

perm_importance = pd.Series(result.importances_mean, index=X.columns)
perm_importance.sort_values(ascending=False)

FIN           0.690141
EQUITY        0.656609
STOCKS        0.125357
LIQ           0.094330
INCOME        0.061073
AGE           0.044654
NFIN          0.041484
EDUC          0.030573
BUS           0.019155
DEBT          0.014345
OCCAT1        0.008780
SPENDMOR      0.007249
FINLIT        0.007189
KIDS          0.005329
MARRIED       0.005119
EMERGSAV      0.002434
HHOUSES       0.002131
LATE          0.001510
YESFINRISK    0.000883
dtype: float64

In [8]:
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

pd.Series(lasso.coef_, index=X.columns)

/Users/jonathansweeney/MIS581/CapstoneProject/.venv-1/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.416e+02, tolerance: 1.749e+01
  model = cd_fast.enet_coordinate_descent(


INCOME        1.200908e-08
FIN          -3.319076e-10
NFIN          2.905410e-09
DEBT          5.452517e-08
EQUITY        1.466264e-08
STOCKS       -1.413196e-08
HHOUSES       2.202834e+00
LIQ           3.392696e-09
YESFINRISK    0.000000e+00
SPENDMOR     -3.276866e-02
LATE         -0.000000e+00
EMERGSAV      1.630636e+00
FINLIT        8.683352e-01
AGE           2.242151e-02
EDUC          7.006229e-01
MARRIED      -1.608786e+00
KIDS         -9.061243e-02
OCCAT1       -6.046287e-01
BUS          -4.828507e-09
dtype: float64

In [9]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, max_iter=10000))
])

pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True


In [10]:
lasso = pipeline.named_steps['lasso']
pd.Series(lasso.coef_, index=X.columns).sort_values()

MARRIED      -0.863085
OCCAT1       -0.502206
KIDS         -0.157651
LATE         -0.146753
SPENDMOR     -0.020831
YESFINRISK    0.000000
LIQ           0.000000
BUS           0.000000
STOCKS        0.000000
EQUITY        0.000000
NFIN          0.000000
DEBT          0.039641
FIN           0.081585
INCOME        0.117711
AGE           0.132154
FINLIT        0.718329
EMERGSAV      0.933144
HHOUSES       1.191696
EDUC          1.830399
dtype: float64

In [18]:
vars_to_keep = ['LOG_RETQLIQ', 'EQUITY', 'FIN', 'LIQ', 'INCOME', 'EDUC', 'HHOUSES', 'EMERGSAV', 'FINLIT', 'DEBT', 'STOCKS']

In [19]:
df_new = df[vars_to_keep]

In [20]:
corr = df_new.corr()
corr['LOG_RETQLIQ'].sort_values(ascending=False)

LOG_RETQLIQ    1.000000
EDUC           0.499085
HHOUSES        0.430080
EMERGSAV       0.407986
FINLIT         0.370634
INCOME         0.121472
FIN            0.116787
DEBT           0.103407
EQUITY         0.097160
LIQ            0.081561
STOCKS         0.056341
Name: LOG_RETQLIQ, dtype: float64

In [21]:
df_new.describe()

,LOG_RETQLIQ,EQUITY,FIN,LIQ,INCOME,EDUC,HHOUSES,EMERGSAV,FINLIT,DEBT,STOCKS
count,4595.000000,4.595000e+03,4.595000e+03,4.595000e+03,4.595000e+03,4595.000000,4595.000000,4595.000000,4595.000000,4.595000e+03,4.595000e+03
mean,7.145172,5.755701e+06,8.323474e+06,7.702663e+05,1.593197e+06,10.327965,0.677040,0.492274,2.303591,3.653292e+05,3.274774e+06
std,6.169690,5.471745e+07,6.379025e+07,8.239290e+06,1.233521e+07,2.816657,0.467658,0.499995,0.829303,2.630899e+06,4.898205e+07
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,0.000000,0.000000e+00,5.290000e+03,1.935000e+03,4.215556e+04,8.000000,0.000000,0.000000,2.000000,0.000000e+00,0.000000e+00
50%,9.798183,1.690000e+04,9.640000e+04,1.460000e+04,9.403933e+04,11.000000,1.000000,0.000000,3.000000,2.900000e+04,0.000000e+00
75%,12.765691,4.750000e+05,1.006315e+06,9.960000e+04,2.648234e+05,12.000000,1.000000,1.000000,3.000000,2.162750e+05,3.900000e+03
max,18.821267,1.788484e+09,1.924763e+09,2.676600e+08,4.532588e+08,14.000000,1.000000,1.000000,3.000000,1.238900e+08,1.786080e+09


In [22]:
df_new['LOG_EQUITY'] = np.log1p(df['EQUITY'])
df_new['LOG_FIN'] = np.log1p(df['FIN'])
df_new['LOG_LIQ'] = np.log1p(df['LIQ'])
df_new['LOG_INCOME'] = np.log1p(df['INCOME'])
df_new['LOG_DEBT'] = np.log1p(df['DEBT'])
df_new['LOG_STOCKS'] = np.log1p(df['STOCKS'])

In [23]:
df_new.columns

Index(['LOG_RETQLIQ', 'EQUITY', 'FIN', 'LIQ', 'INCOME', 'EDUC', 'HHOUSES',
       'EMERGSAV', 'FINLIT', 'DEBT', 'STOCKS', 'LOG_EQUITY', 'LOG_FIN',
       'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT', 'LOG_STOCKS'],
      dtype='str')

In [24]:
df_new.drop(columns=['EQUITY', 'FIN', 'LIQ', 'INCOME', 'DEBT', 'STOCKS'], inplace=True)

In [25]:
df_new.columns

Index(['LOG_RETQLIQ', 'EDUC', 'HHOUSES', 'EMERGSAV', 'FINLIT', 'LOG_EQUITY',
       'LOG_FIN', 'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT', 'LOG_STOCKS'],
      dtype='str')

In [26]:
df_new.describe()

,LOG_RETQLIQ,EDUC,HHOUSES,EMERGSAV,FINLIT,LOG_EQUITY,LOG_FIN,LOG_LIQ,LOG_INCOME,LOG_DEBT,LOG_STOCKS
count,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000
mean,7.145172,10.327965,0.677040,0.492274,2.303591,7.695140,11.136226,9.406495,11.674989,8.154658,3.459225
std,6.169690,2.816657,0.467658,0.499995,0.829303,6.369022,3.855917,3.169348,2.007434,5.290533,5.700275
min,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,8.000000,0.000000,0.000000,2.000000,0.000000,8.573761,7.568349,10.649146,0.000000,0.000000
50%,9.798183,11.000000,1.000000,0.000000,3.000000,9.735128,11.476272,9.588845,11.451479,10.275086,0.000000
75%,12.765691,12.000000,1.000000,1.000000,3.000000,13.071072,13.821806,11.508927,12.486822,12.284310,8.268988
max,18.821267,14.000000,1.000000,1.000000,3.000000,21.304634,21.378069,19.405228,19.931974,18.634905,21.303289


In [27]:
corr = df_new.corr(numeric_only=True)
corr['LOG_RETQLIQ'].sort_values(ascending=False)

LOG_RETQLIQ    1.000000
LOG_EQUITY     0.843514
LOG_FIN        0.746423
LOG_LIQ        0.582779
EDUC           0.499085
LOG_INCOME     0.498783
HHOUSES        0.430080
LOG_STOCKS     0.422224
EMERGSAV       0.407986
FINLIT         0.370634
LOG_DEBT       0.141342
Name: LOG_RETQLIQ, dtype: float64

In [28]:
X = df_new.drop(columns=['LOG_RETQLIQ'])
y = df_new['LOG_RETQLIQ']

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

LOG_EQUITY    0.729250
LOG_FIN       0.097199
LOG_LIQ       0.049060
LOG_STOCKS    0.039220
LOG_INCOME    0.038233
LOG_DEBT      0.020289
EDUC          0.014770
FINLIT        0.005964
EMERGSAV      0.003014
HHOUSES       0.003001
dtype: float64

In [29]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, max_iter=10000))
])

pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True


In [30]:
lasso = pipeline.named_steps['lasso']
pd.Series(lasso.coef_, index=X.columns).sort_values()

LOG_STOCKS   -0.709162
EMERGSAV     -0.000000
LOG_LIQ      -0.000000
LOG_INCOME    0.000000
HHOUSES       0.044562
FINLIT        0.091121
EDUC          0.092316
LOG_DEBT      0.150666
LOG_FIN       0.720925
LOG_EQUITY    4.800250
dtype: float64

In [31]:
df_new.to_csv('../data/processed/Cleaned_SCF_2022.csv')

In [32]:
df_new.shape

(4595, 11)